# BE-Net patient pipeline (research only)
Run cells in order. This notebook has not trained a model. Predictions are not validated clinical diagnoses. Read README.md before use.

**Privacy:** Do not upload patient data to Colab unless your organization and data permissions explicitly authorize it. Keep data and outputs private. Local execution is available with the same commands. The supplied local cohort has additional label IDs requiring a verified policy; the pipeline intentionally stops rather than guessing a mapping.

## 1. Enable GPU and add code
In Colab choose Runtime > Change runtime type > GPU. Upload only the pdi_pipeline code folder (or its ZIP) using the Files pane, then place it under /content/project. Keep the Python package directory named pdi_pipeline. Extract code ZIPs using the Files pane or your trusted local workflow. Do not upload data in this step.

In [ ]:
%cd /content/project
!nvidia-smi
# Install the pinned baseline on Python 3.11/3.12. Restart runtime if imports request it.
!python -m pip install -r pdi_pipeline/requirements.txt
!python -m pip freeze > colab_environment.txt

## 2. Set paths
After privacy authorization, make the approved dataset accessible to this runtime using your approved storage process. No mounting/upload is automated here. Use one extracted folder per patient, containing cube.dat, cube.hdr and one *_GT.mat. If you have archives, run the optional importer. Set PREPROCESSING to the verified upstream preprocessing protocol. Different values are deliberately incompatible.

In [ ]:
import json
from pathlib import Path
DATASET_ROOT = "/content/private_data"
OUTPUT = "/content/private_runs/full"
PREPROCESSING = "raw_ENVI_values_no_spectral_reordering"
cfg = json.loads(Path("pdi_pipeline/config.example.json").read_text())
cfg.update(dataset_root=DATASET_ROOT, output=OUTPUT, preprocessing=PREPROCESSING)
Path("pdi_pipeline/config.json").write_text(json.dumps(cfg, indent=2))
CONFIG = "pdi_pipeline/config.json"
print("GPU training uses CUDA automatically when available.")

In [ ]:
# OPTIONAL: import local runtime ZIPs, after authorized transfer. Uncomment and edit.
# !python -m pdi_pipeline.import_archives --source /content/private_archives --destination /content/private_data

## 3. Inspect, review overlays, then split
Inspect validates all patient files and produces inspection PNGs. Open every overlay in the Files pane and verify spatial orientation using known landmarks. Shape equality alone cannot establish registration. Resolve missing patients and label policy before preparation; do not discard additional labels or invent clinical meanings. prepare requires 77 unique patients, creates 53/12/12 split and training-only normalization. It reports class coverage.

In [ ]:
!python -m pdi_pipeline.cli inspect --config {CONFIG}
# STOP and review the inspection report and every overlay before the next cell.

In [ ]:
!python -m pdi_pipeline.cli prepare --config {CONFIG}
!python -m pdi_pipeline.checks

## 4. Short check in separate output
This uses two training batches and full validation. It checks execution only and is not a valid experiment result.

In [ ]:
smoke = dict(cfg, output="/content/private_runs/smoke", epochs=1)
Path("pdi_pipeline/config.smoke.json").write_text(json.dumps(smoke, indent=2))
!python -m pdi_pipeline.cli prepare --config pdi_pipeline/config.smoke.json
!python -m pdi_pipeline.cli train --config pdi_pipeline/config.smoke.json --max-steps 2

## 5. Full training and resume
Start at patch_size=64 and batch_size=2. Edit epochs/learning_rate before training if needed. Save outputs to approved persistent private storage; Colab runtime disks are temporary. best_model.pt is selected by pooled validation mean IoU, last_model.pt supports resuming, and epoch checkpoints are saved every five epochs. Epoch checkpoints are versions of the same architecture.

In [ ]:
!python -m pdi_pipeline.cli train --config {CONFIG}
# To resume an interrupted run, use this command instead:
# !python -m pdi_pipeline.cli train --config {CONFIG} --resume {OUTPUT}/last_model.pt

In [ ]:
from IPython.display import display, Image
display(Image(filename=f"{OUTPUT}/curves.png"))
!python -m pdi_pipeline.cli validate --config {CONFIG} --checkpoint {OUTPUT}/best_model.pt
!python -m pdi_pipeline.cli compare --config {CONFIG}

## 6. Final test, only after configuration is finalized
Use validation only for settings/checkpoint decisions. Run the next cell once after those decisions are complete. Do not change settings in response to test results. Reports separate per-patient, pooled-pixel and patient-macro results; undefined metrics are null, and label 0 is ignored.

In [ ]:
# Run deliberately only after final decisions:
# !python -m pdi_pipeline.cli finalize --config {CONFIG}
# !python -m pdi_pipeline.cli test --config {CONFIG} --checkpoint {OUTPUT}/best_model.pt

## 7. Inference without annotations
Edit the new patient path. Its header must have the exact trained spectral ordering/units and verified same preprocessing. Ground truth is optional and used only for comparison after prediction. Copy exports to the existing PDI Load prediction mask button. GUI inference execution has not been wired in.

In [ ]:
NEW_CUBE = "/content/private_new_patient/cube.dat"
PREDICTION_OUTPUT = "/content/private_predictions/new_patient"
# Uncomment after setting real paths:
# !python -m pdi_pipeline.cli predict --config {CONFIG} --checkpoint {OUTPUT}/best_model.pt --cube {NEW_CUBE} --prediction-output {PREDICTION_OUTPUT}
# Optional comparison: append --gt /path/to/NewPatient_GT.mat